# WikiArt Multi-Task — Test 9 (Style + Artist + Emotion)

This test trains **one shared model** to predict:
- style (WikiArt)
- artist (WikiArt)
- emotion (ArtEmis mapped to WikiArt images)

Goal: use shared visual features so tasks learn from each other and improve total quality.

### Data cleaning strategy
- Match WikiArt images by normalized `(style, painting_name)` keys
- Keep only rows where image file exists
- Aggregate multiple ArtEmis emotion rows per image using majority vote
- Train with style + artist always, emotion only when available

In [ ]:
import copy
import math
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

try:
    import timm
except ImportError as exc:
    raise ImportError("This notebook requires timm. Install with: pip install timm") from exc


@dataclass
class TrainConfig:
    model_name: str = "vit_base_patch16_384.augreg_in21k_ft_in1k"
    image_size: int = 384
    batch_size: int = 8
    effective_batch_size: int = 64
    head_epochs: int = 4
    ft_epochs: int = 56
    patience: int = 16
    warmup_epochs: int = 4
    head_lr: float = 4e-4
    ft_backbone_lr: float = 4e-6
    ft_head_lr: float = 1.8e-5
    weight_decay: float = 1.2e-4
    mixup_alpha: float = 0.6
    cutmix_alpha: float = 1.0
    mix_probability: float = 0.9
    label_smoothing: float = 0.08
    ema_decay: float = 0.99985
    grad_clip_norm: float = 1.0
    tta: bool = True
    use_weighted_sampler: bool = True
    seed: int = 42
    num_workers: int = 0
    progress_every: int = 300
    emotion_loss_weight: float = 0.7


class MultiTaskDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, image_root: Path, transform=None):
        self.rows = rows.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int):
        row = self.rows.iloc[idx]
        image_path = self.image_root / row["relative_path"]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        style_label = int(row["style_label"])
        artist_label = int(row["artist_label"])

        if pd.isna(row["emotion_label"]):
            emotion_label = -1
            has_emotion = 0
        else:
            emotion_label = int(row["emotion_label"])
            has_emotion = 1

        return image, style_label, artist_label, emotion_label, has_emotion


class MultiTaskModel(nn.Module):
    def __init__(self, model_name: str, n_style: int, n_artist: int, n_emotion: int):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        feat_dim = self.backbone.num_features
        self.style_head = nn.Linear(feat_dim, n_style)
        self.artist_head = nn.Linear(feat_dim, n_artist)
        self.emotion_head = nn.Linear(feat_dim, n_emotion)

    def forward(self, x):
        feat = self.backbone(x)
        style_logits = self.style_head(feat)
        artist_logits = self.artist_head(feat)
        emotion_logits = self.emotion_head(feat)
        return style_logits, artist_logits, emotion_logits


class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.9997):
        self.decay = decay
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model: nn.Module):
        msd = model.state_dict()
        for k, v in self.ema_model.state_dict().items():
            if k in msd:
                model_v = msd[k].detach()
                if torch.is_floating_point(v):
                    v.mul_(self.decay).add_(model_v, alpha=1.0 - self.decay)
                else:
                    v.copy_(model_v)


def normalize_text(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "", value)
    return value


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def discover_paths(project_root: Path):
    wikiart_dir = project_root / "datasets" / "Wikiart"
    style_train_csv = wikiart_dir / "style_train.csv"
    style_val_csv = wikiart_dir / "style_val.csv"
    artist_train_csv = wikiart_dir / "artist_train.csv"
    artist_val_csv = wikiart_dir / "artist_val.csv"

    artemis_csv = project_root / "datasets" / "ArtEmis" / "Contrastive.csv"

    required = [style_train_csv, style_val_csv, artist_train_csv, artist_val_csv, artemis_csv]
    for p in required:
        if not p.exists():
            raise FileNotFoundError(f"Required file missing: {p}")

    return wikiart_dir, style_train_csv, style_val_csv, artist_train_csv, artist_val_csv, artemis_csv


def load_label_csv(path: Path, col_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=["relative_path", col_name])
    df[col_name] = df[col_name].astype(int)
    return df


def filter_existing_rows(df: pd.DataFrame, image_root: Path, split_name: str) -> pd.DataFrame:
    keep_mask = df["relative_path"].map(lambda p: (image_root / p).exists())
    missing_count = int((~keep_mask).sum())
    if missing_count > 0:
        print(f"[{split_name}] Filtered out {missing_count} missing files.")
    return df.loc[keep_mask].reset_index(drop=True)


def merge_style_artist(style_df: pd.DataFrame, artist_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    merged = style_df.merge(artist_df, on="relative_path", how="inner")
    print(f"[{split_name}] style+artist merged rows: {len(merged)}")
    return merged


def build_wikiart_lookup(all_paths: pd.Series) -> Dict[Tuple[str, str], str]:
    lookup = {}
    for rp in all_paths.dropna().astype(str).tolist():
        p = Path(rp)
        style_key = normalize_text(p.parent.name)
        painting_key = normalize_text(p.stem)
        key = (style_key, painting_key)
        if key not in lookup:
            lookup[key] = rp
    return lookup


def build_emotion_map(artemis_csv: Path, wikiart_lookup: Dict[Tuple[str, str], str]) -> Tuple[pd.DataFrame, Dict[str, int]]:
    artemis_df = pd.read_csv(artemis_csv)
    required_cols = ["emotion", "art_style", "painting"]
    for col in required_cols:
        if col not in artemis_df.columns:
            raise ValueError(f"ArtEmis file missing column: {col}")

    artemis_df = artemis_df.dropna(subset=required_cols).copy()

    matched_paths = []
    matched_emotions = []
    for _, row in artemis_df.iterrows():
        style_key = normalize_text(row["art_style"])
        painting_key = normalize_text(row["painting"])
        key = (style_key, painting_key)
        rp = wikiart_lookup.get(key)
        if rp is not None:
            matched_paths.append(rp)
            matched_emotions.append(str(row["emotion"]).strip().lower())

    if len(matched_paths) == 0:
        raise RuntimeError("No ArtEmis rows could be matched to WikiArt paths.")

    tmp = pd.DataFrame({"relative_path": matched_paths, "emotion": matched_emotions})

    # Majority vote per image
    image_to_emotion = {}
    for rp, grp in tmp.groupby("relative_path"):
        cnt = Counter(grp["emotion"].tolist())
        image_to_emotion[rp] = cnt.most_common(1)[0][0]

    unique_emotions = sorted(set(image_to_emotion.values()))
    emotion_to_idx = {emo: i for i, emo in enumerate(unique_emotions)}

    out = pd.DataFrame({
        "relative_path": list(image_to_emotion.keys()),
        "emotion_name": list(image_to_emotion.values()),
    })
    out["emotion_label"] = out["emotion_name"].map(emotion_to_idx).astype(int)

    print(f"ArtEmis matched images: {len(out)} | emotion classes: {len(emotion_to_idx)}")
    return out, emotion_to_idx


def stratified_half_split(df: pd.DataFrame, label_col: str, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    sampled_parts = []
    for _, grp in df.groupby(label_col, sort=False):
        if len(grp) > 1:
            sampled_parts.append(grp.sample(frac=0.5, random_state=seed))
        else:
            sampled_parts.append(grp)

    right = pd.concat(sampled_parts, axis=0)
    left = df.drop(index=right.index)

    if len(left) == 0:
        left = df.sample(frac=0.5, random_state=seed)
        right = df.drop(index=left.index)

    return left.reset_index(drop=True), right.reset_index(drop=True)


def build_transforms(image_size: int):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.55, 1.0), ratio=(0.75, 1.33)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandAugment(num_ops=2, magnitude=11),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.04),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.18), ratio=(0.3, 3.3), value="random"),
    ])
    eval_tfms = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return train_tfms, eval_tfms


def create_weighted_sampler(style_labels: np.ndarray) -> WeightedRandomSampler:
    class_counts = np.bincount(style_labels)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = class_weights[style_labels]
    sample_weights = np.sqrt(sample_weights)
    sample_weights = sample_weights / sample_weights.mean()
    return WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )


def mixup_cutmix(inputs, targets, alpha_mixup, alpha_cutmix, p, num_classes):
    if np.random.rand() > p:
        return inputs, torch.nn.functional.one_hot(targets, num_classes=num_classes).float()

    batch_size = inputs.size(0)
    indices = torch.randperm(batch_size, device=inputs.device)
    targets_onehot = torch.nn.functional.one_hot(targets, num_classes=num_classes).float()
    shuffled_targets = targets_onehot[indices]

    use_cutmix = alpha_cutmix > 0 and np.random.rand() < 0.5
    if use_cutmix:
        lam = np.random.beta(alpha_cutmix, alpha_cutmix)
        h, w = inputs.size(2), inputs.size(3)
        cut_rat = np.sqrt(1.0 - lam)
        cut_w = int(w * cut_rat)
        cut_h = int(h * cut_rat)

        cx = np.random.randint(w)
        cy = np.random.randint(h)

        x1 = np.clip(cx - cut_w // 2, 0, w)
        x2 = np.clip(cx + cut_w // 2, 0, w)
        y1 = np.clip(cy - cut_h // 2, 0, h)
        y2 = np.clip(cy + cut_h // 2, 0, h)

        mixed = inputs.clone()
        mixed[:, :, y1:y2, x1:x2] = inputs[indices, :, y1:y2, x1:x2]
        lam_adjusted = 1.0 - ((x2 - x1) * (y2 - y1) / (w * h))
        soft_targets = targets_onehot * lam_adjusted + shuffled_targets * (1.0 - lam_adjusted)
        return mixed, soft_targets

    lam = np.random.beta(alpha_mixup, alpha_mixup) if alpha_mixup > 0 else 1.0
    mixed = lam * inputs + (1.0 - lam) * inputs[indices]
    soft_targets = lam * targets_onehot + (1.0 - lam) * shuffled_targets
    return mixed, soft_targets


def top1_accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
    pred = logits.argmax(dim=1)
    return float((pred == targets).float().mean().item())


def freeze_backbone(model: MultiTaskModel, freeze: bool):
    for p in model.backbone.parameters():
        p.requires_grad = not freeze
    for p in model.style_head.parameters():
        p.requires_grad = True
    for p in model.artist_head.parameters():
        p.requires_grad = True
    for p in model.emotion_head.parameters():
        p.requires_grad = True


def get_trainable_params(model: MultiTaskModel, backbone_lr: float, head_lr: float, wd: float):
    backbone = [p for p in model.backbone.parameters() if p.requires_grad]
    heads = [
        p for p in list(model.style_head.parameters()) + list(model.artist_head.parameters()) + list(model.emotion_head.parameters())
        if p.requires_grad
    ]
    groups = []
    if backbone:
        groups.append({"params": backbone, "lr": backbone_lr, "weight_decay": wd})
    if heads:
        groups.append({"params": heads, "lr": head_lr, "weight_decay": wd})
    return groups


def build_scheduler(optimizer, warmup_epochs: int, total_epochs: int):
    warmup_epochs = max(0, warmup_epochs)
    total_epochs = max(1, total_epochs)
    if warmup_epochs > 0:
        warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.12, total_iters=warmup_epochs)
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_epochs - warmup_epochs))
        return torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_epochs])
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)


def run_epoch(model, loader, optimizer, scaler, device, cfg, n_style, n_artist, n_emotion, is_train=True, accum_steps=1):
    if is_train:
        model.train()
    else:
        model.eval()

    ce_style = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    ce_artist = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    ce_emotion = nn.CrossEntropyLoss()

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    total_samples = 0

    style_acc_sum = 0.0
    artist_acc_sum = 0.0

    emotion_correct = 0
    emotion_total = 0

    for step, batch in enumerate(loader, start=1):
        images, style_y, artist_y, emotion_y, emotion_mask = batch
        images = images.to(device, non_blocking=True)
        style_y = style_y.to(device, non_blocking=True)
        artist_y = artist_y.to(device, non_blocking=True)
        emotion_y = emotion_y.to(device, non_blocking=True)
        emotion_mask = emotion_mask.to(device, non_blocking=True).bool()

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            if is_train:
                mixed_images, style_soft = mixup_cutmix(images, style_y, cfg.mixup_alpha, cfg.cutmix_alpha, cfg.mix_probability, n_style)
                _, artist_soft = mixup_cutmix(images, artist_y, cfg.mixup_alpha, cfg.cutmix_alpha, cfg.mix_probability, n_artist)
                style_logits, artist_logits, emotion_logits = model(mixed_images)

                style_loss = torch.mean(torch.sum(-style_soft * torch.log_softmax(style_logits, dim=1), dim=1))
                artist_loss = torch.mean(torch.sum(-artist_soft * torch.log_softmax(artist_logits, dim=1), dim=1))
            else:
                style_logits, artist_logits, emotion_logits = model(images)
                style_loss = ce_style(style_logits, style_y)
                artist_loss = ce_artist(artist_logits, artist_y)

            if emotion_mask.any():
                emotion_loss = ce_emotion(emotion_logits[emotion_mask], emotion_y[emotion_mask])
            else:
                emotion_loss = torch.zeros((), device=device)

            loss = style_loss + artist_loss + cfg.emotion_loss_weight * emotion_loss
            loss = loss / accum_steps

        if is_train:
            scaler.scale(loss).backward()
            if step % accum_steps == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        bs = images.size(0)
        total_samples += bs
        total_loss += float(loss.item()) * accum_steps * bs

        with torch.no_grad():
            style_acc = top1_accuracy(style_logits, style_y)
            artist_acc = top1_accuracy(artist_logits, artist_y)
            style_acc_sum += style_acc * bs
            artist_acc_sum += artist_acc * bs

            if emotion_mask.any():
                e_pred = emotion_logits[emotion_mask].argmax(dim=1)
                e_true = emotion_y[emotion_mask]
                emotion_correct += int((e_pred == e_true).sum().item())
                emotion_total += int(e_true.numel())

        if is_train and (step % cfg.progress_every == 0 or step == len(loader)):
            print(f"  [train] batch {step}/{len(loader)} | avg_loss={total_loss/max(1,total_samples):.4f} | style={style_acc_sum/max(1,total_samples):.4f} | artist={artist_acc_sum/max(1,total_samples):.4f}")

    out = {
        "loss": total_loss / max(1, total_samples),
        "style_acc": style_acc_sum / max(1, total_samples),
        "artist_acc": artist_acc_sum / max(1, total_samples),
        "emotion_acc": (emotion_correct / emotion_total) if emotion_total > 0 else float("nan"),
        "emotion_count": emotion_total,
    }
    return out


@torch.no_grad()
def evaluate_with_tta(model, loader, device, cfg, n_style, n_artist, n_emotion):
    model.eval()

    ce_style = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    ce_artist = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    ce_emotion = nn.CrossEntropyLoss()

    total_loss = 0.0
    total_samples = 0

    style_acc_sum = 0.0
    artist_acc_sum = 0.0
    emotion_correct = 0
    emotion_total = 0

    for batch in loader:
        images, style_y, artist_y, emotion_y, emotion_mask = batch
        images = images.to(device, non_blocking=True)
        style_y = style_y.to(device, non_blocking=True)
        artist_y = artist_y.to(device, non_blocking=True)
        emotion_y = emotion_y.to(device, non_blocking=True)
        emotion_mask = emotion_mask.to(device, non_blocking=True).bool()

        style_logits, artist_logits, emotion_logits = model(images)
        if cfg.tta:
            s2, a2, e2 = model(torch.flip(images, dims=[3]))
            style_logits = (style_logits + s2) / 2.0
            artist_logits = (artist_logits + a2) / 2.0
            emotion_logits = (emotion_logits + e2) / 2.0

        style_loss = ce_style(style_logits, style_y)
        artist_loss = ce_artist(artist_logits, artist_y)
        if emotion_mask.any():
            emotion_loss = ce_emotion(emotion_logits[emotion_mask], emotion_y[emotion_mask])
        else:
            emotion_loss = torch.zeros((), device=device)

        loss = style_loss + artist_loss + cfg.emotion_loss_weight * emotion_loss

        bs = images.size(0)
        total_samples += bs
        total_loss += float(loss.item()) * bs

        style_acc_sum += top1_accuracy(style_logits, style_y) * bs
        artist_acc_sum += top1_accuracy(artist_logits, artist_y) * bs

        if emotion_mask.any():
            e_pred = emotion_logits[emotion_mask].argmax(dim=1)
            e_true = emotion_y[emotion_mask]
            emotion_correct += int((e_pred == e_true).sum().item())
            emotion_total += int(e_true.numel())

    return {
        "loss": total_loss / max(1, total_samples),
        "style_acc": style_acc_sum / max(1, total_samples),
        "artist_acc": artist_acc_sum / max(1, total_samples),
        "emotion_acc": (emotion_correct / emotion_total) if emotion_total > 0 else float("nan"),
        "emotion_count": emotion_total,
    }


def fit(project_root: Path, cfg: TrainConfig):
    set_seed(cfg.seed)
    Image.MAX_IMAGE_PIXELS = None

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    wikiart_dir, style_train_csv, style_val_csv, artist_train_csv, artist_val_csv, artemis_csv = discover_paths(project_root)

    style_train = filter_existing_rows(load_label_csv(style_train_csv, "style_label"), wikiart_dir, "style_train")
    style_val = filter_existing_rows(load_label_csv(style_val_csv, "style_label"), wikiart_dir, "style_val")
    artist_train = filter_existing_rows(load_label_csv(artist_train_csv, "artist_label"), wikiart_dir, "artist_train")
    artist_val = filter_existing_rows(load_label_csv(artist_val_csv, "artist_label"), wikiart_dir, "artist_val")

    train_core = merge_style_artist(style_train, artist_train, "train")
    val_core = merge_style_artist(style_val, artist_val, "val")

    all_paths = pd.concat([train_core["relative_path"], val_core["relative_path"]], axis=0).reset_index(drop=True)
    wikiart_lookup = build_wikiart_lookup(all_paths)

    emotion_df, emotion_to_idx = build_emotion_map(artemis_csv, wikiart_lookup)

    train_df = train_core.merge(emotion_df[["relative_path", "emotion_label"]], on="relative_path", how="left")
    val_merged = val_core.merge(emotion_df[["relative_path", "emotion_label"]], on="relative_path", how="left")

    eval_val_df, eval_test_df = stratified_half_split(val_merged, label_col="style_label", seed=cfg.seed)

    n_style = int(max(train_df["style_label"].max(), eval_val_df["style_label"].max(), eval_test_df["style_label"].max()) + 1)
    n_artist = int(max(train_df["artist_label"].max(), eval_val_df["artist_label"].max(), eval_test_df["artist_label"].max()) + 1)
    n_emotion = len(emotion_to_idx)

    train_tfms, eval_tfms = build_transforms(cfg.image_size)

    train_ds = MultiTaskDataset(train_df, wikiart_dir, transform=train_tfms)
    val_ds = MultiTaskDataset(eval_val_df, wikiart_dir, transform=eval_tfms)
    test_ds = MultiTaskDataset(eval_test_df, wikiart_dir, transform=eval_tfms)

    sampler = create_weighted_sampler(train_df["style_label"].values) if cfg.use_weighted_sampler else None
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler, shuffle=(sampler is None), num_workers=cfg.num_workers, pin_memory=(device.type=="cuda"), drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

    print(f"Device: {device}")
    print(f"Model: {cfg.model_name}")
    print(f"Train/Val/Test sizes: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")
    print(f"Classes -> style:{n_style}, artist:{n_artist}, emotion:{n_emotion}")
    print(f"Emotion coverage train/val/test: {train_df['emotion_label'].notna().mean():.3f}/{eval_val_df['emotion_label'].notna().mean():.3f}/{eval_test_df['emotion_label'].notna().mean():.3f}")

    model = MultiTaskModel(cfg.model_name, n_style=n_style, n_artist=n_artist, n_emotion=n_emotion).to(device)
    scaler = torch.amp.GradScaler(device="cuda", enabled=(device.type == "cuda"))
    ema = ModelEMA(model, decay=cfg.ema_decay)

    grad_accum_steps = max(1, math.ceil(cfg.effective_batch_size / cfg.batch_size))

    history = []
    best_score = -1.0
    best_epoch = -1
    patience_left = cfg.patience

    models_dir = project_root / "models"
    results_dir = project_root / "models" / "results"
    models_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = models_dir / "wikiart_test9_multitask_best.pt"

    epoch_counter = 0

    for stage_name, stage_epochs, freeze_flag, bb_lr, hd_lr in [
        ("head-only", cfg.head_epochs, True, cfg.head_lr, cfg.head_lr),
        ("fine-tune", cfg.ft_epochs, False, cfg.ft_backbone_lr, cfg.ft_head_lr),
    ]:
        freeze_backbone(model, freeze=freeze_flag)
        optimizer = torch.optim.AdamW(get_trainable_params(model, bb_lr, hd_lr, cfg.weight_decay))
        scheduler = build_scheduler(optimizer, min(cfg.warmup_epochs, stage_epochs), stage_epochs)

        for _ in range(stage_epochs):
            epoch_counter += 1
            print(f"\n--- Epoch {epoch_counter:02d} / {cfg.head_epochs + cfg.ft_epochs} ({stage_name}) ---")

            train_metrics = run_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                scaler=scaler,
                device=device,
                cfg=cfg,
                n_style=n_style,
                n_artist=n_artist,
                n_emotion=n_emotion,
                is_train=True,
                accum_steps=grad_accum_steps,
            )

            if ema is not None:
                ema.update(model)
                eval_model = ema.ema_model
            else:
                eval_model = model

            val_metrics = evaluate_with_tta(eval_model, val_loader, device, cfg, n_style, n_artist, n_emotion)
            scheduler.step()

            # Combined score (style strongest, then artist, then emotion)
            emotion_acc_for_score = 0.0 if np.isnan(val_metrics['emotion_acc']) else val_metrics['emotion_acc']
            combined_score = 0.45 * val_metrics['style_acc'] + 0.35 * val_metrics['artist_acc'] + 0.20 * emotion_acc_for_score

            lr_now = optimizer.param_groups[0]['lr']
            history.append({
                'epoch': epoch_counter,
                'stage': stage_name,
                'lr': lr_now,
                'train_loss': train_metrics['loss'],
                'train_style_acc': train_metrics['style_acc'],
                'train_artist_acc': train_metrics['artist_acc'],
                'train_emotion_acc': train_metrics['emotion_acc'],
                'val_loss': val_metrics['loss'],
                'val_style_acc': val_metrics['style_acc'],
                'val_artist_acc': val_metrics['artist_acc'],
                'val_emotion_acc': val_metrics['emotion_acc'],
                'val_combined_score': combined_score,
            })

            print(
                f"Epoch {epoch_counter:02d} | lr={lr_now:.2e} | "
                f"train(style/artist/emotion)=({train_metrics['style_acc']:.4f}/{train_metrics['artist_acc']:.4f}/{train_metrics['emotion_acc'] if not np.isnan(train_metrics['emotion_acc']) else float('nan'):.4f}) | "
                f"val(style/artist/emotion)=({val_metrics['style_acc']:.4f}/{val_metrics['artist_acc']:.4f}/{val_metrics['emotion_acc'] if not np.isnan(val_metrics['emotion_acc']) else float('nan'):.4f}) | "
                f"combined={combined_score:.4f}"
            )

            if combined_score > best_score:
                best_score = combined_score
                best_epoch = epoch_counter
                patience_left = cfg.patience
                torch.save({
                    'model_state': copy.deepcopy(eval_model.state_dict()),
                    'config': asdict(cfg),
                    'best_combined_score': best_score,
                    'best_epoch': best_epoch,
                    'n_style': n_style,
                    'n_artist': n_artist,
                    'n_emotion': n_emotion,
                    'emotion_to_idx': emotion_to_idx,
                    'model_name': cfg.model_name,
                    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
                }, checkpoint_path)
            else:
                patience_left -= 1

            if patience_left <= 0:
                print(f"Early stopping at epoch {epoch_counter}. Best epoch: {best_epoch}")
                break

        if patience_left <= 0:
            break

    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state['model_state'])

    final_val = evaluate_with_tta(model, val_loader, device, cfg, n_style, n_artist, n_emotion)
    final_test = evaluate_with_tta(model, test_loader, device, cfg, n_style, n_artist, n_emotion)

    history_df = pd.DataFrame(history)
    history_csv = results_dir / "wikiart_test9_multitask_history.csv"
    history_df.to_csv(history_csv, index=False)

    summary_row = {
        'notebook': 'wikiart_style_classification_test9_multitask.ipynb',
        'exists': True,
        'model_name': cfg.model_name,
        'val_style_acc': final_val['style_acc'],
        'val_artist_acc': final_val['artist_acc'],
        'val_emotion_acc': final_val['emotion_acc'],
        'test_style_acc': final_test['style_acc'],
        'test_artist_acc': final_test['artist_acc'],
        'test_emotion_acc': final_test['emotion_acc'],
        'best_combined_score': best_score,
        'best_epoch': best_epoch,
        'experiment': 'test9',
        'saved_at_utc': datetime.now(timezone.utc).isoformat(),
    }

    summary_csv = results_dir / "wikiart_tests_1_to_9_summary.csv"
    prior_paths = [
        results_dir / "wikiart_tests_1_to_8_summary.csv",
        results_dir / "wikiart_tests_1_to_7_summary.csv",
        results_dir / "wikiart_tests_1_to_6_summary.csv",
    ]

    rows = []
    for prior in prior_paths:
        if prior.exists():
            rows.extend(pd.read_csv(prior).to_dict(orient="records"))
            break

    rows = [r for r in rows if str(r.get('experiment', '')).lower() != 'test9']
    rows.append(summary_row)
    pd.DataFrame(rows).to_csv(summary_csv, index=False)

    print("\nFinal evaluation (Test 9 multi-task)")
    print(f"Validation -> style: {final_val['style_acc']:.4f}, artist: {final_val['artist_acc']:.4f}, emotion: {final_val['emotion_acc']:.4f}")
    print(f"Test       -> style: {final_test['style_acc']:.4f}, artist: {final_test['artist_acc']:.4f}, emotion: {final_test['emotion_acc']:.4f}")
    print(f"Best combined score: {best_score:.4f} at epoch {best_epoch}")
    print(f"Saved checkpoint: {checkpoint_path}")
    print(f"Saved history: {history_csv}")
    print(f"Saved summary: {summary_csv}")

    return history_df, final_val, final_test

In [ ]:
# --- Test 9 improvements patch ---
# Implements:
# 1) no emotion loss on mixed batches
# 2) emotion label smoothing
# 3) loss normalization by log(num_classes)
# 4) MLP heads per task
# 5) uncertainty-based learnable task weighting
# 6) EMA updates per optimizer step (not once per epoch)


def _soft_target_ce(logits: torch.Tensor, soft_targets: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.sum(-soft_targets * torch.log_softmax(logits, dim=1), dim=1))


class MultiTaskModel(nn.Module):
    def __init__(self, model_name: str, n_style: int, n_artist: int, n_emotion: int):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        feat_dim = self.backbone.num_features

        def make_head(out_dim: int):
            return nn.Sequential(
                nn.Linear(feat_dim, feat_dim),
                nn.LayerNorm(feat_dim),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(feat_dim, out_dim),
            )

        self.style_head = make_head(n_style)
        self.artist_head = make_head(n_artist)
        self.emotion_head = make_head(n_emotion)

        self.log_var_style = nn.Parameter(torch.zeros(1))
        self.log_var_artist = nn.Parameter(torch.zeros(1))
        self.log_var_emotion = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        feat = self.backbone(x)
        style_logits = self.style_head(feat)
        artist_logits = self.artist_head(feat)
        emotion_logits = self.emotion_head(feat)
        return style_logits, artist_logits, emotion_logits


class ModelEMA:
    active_instance = None
    _allow_update = False

    def __init__(self, model: nn.Module, decay: float = 0.9997):
        self.decay = decay
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters():
            p.requires_grad = False
        ModelEMA.active_instance = self

    @torch.no_grad()
    def update(self, model: nn.Module):
        if not ModelEMA._allow_update:
            return

        msd = model.state_dict()
        for k, v in self.ema_model.state_dict().items():
            if k in msd:
                model_v = msd[k].detach()
                if torch.is_floating_point(v):
                    v.mul_(self.decay).add_(model_v, alpha=1.0 - self.decay)
                else:
                    v.copy_(model_v)


def get_trainable_params(model: MultiTaskModel, backbone_lr: float, head_lr: float, wd: float):
    backbone = [p for p in model.backbone.parameters() if p.requires_grad]
    heads = [
        p for p in list(model.style_head.parameters()) + list(model.artist_head.parameters()) + list(model.emotion_head.parameters())
        if p.requires_grad
    ]
    uncertainty = [model.log_var_style, model.log_var_artist, model.log_var_emotion]

    groups = []
    if backbone:
        groups.append({"params": backbone, "lr": backbone_lr, "weight_decay": wd})
    groups.append({"params": heads + uncertainty, "lr": head_lr, "weight_decay": wd})
    return groups


def run_epoch(model, loader, optimizer, scaler, device, cfg, n_style, n_artist, n_emotion, is_train=True, accum_steps=1):
    if is_train:
        model.train()
    else:
        model.eval()

    ce_style = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    ce_artist = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    emotion_ls = float(getattr(cfg, "emotion_label_smoothing", 0.12))
    ce_emotion = nn.CrossEntropyLoss(label_smoothing=emotion_ls)

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    total_samples = 0

    style_acc_sum = 0.0
    artist_acc_sum = 0.0

    emotion_correct = 0
    emotion_total = 0

    style_norm = max(1.0, math.log(max(2, n_style)))
    artist_norm = max(1.0, math.log(max(2, n_artist)))
    emotion_norm = max(1.0, math.log(max(2, n_emotion)))

    for step, batch in enumerate(loader, start=1):
        images, style_y, artist_y, emotion_y, emotion_mask = batch
        images = images.to(device, non_blocking=True)
        style_y = style_y.to(device, non_blocking=True)
        artist_y = artist_y.to(device, non_blocking=True)
        emotion_y = emotion_y.to(device, non_blocking=True)
        emotion_mask = emotion_mask.to(device, non_blocking=True).bool()

        do_mix = is_train and (np.random.rand() < cfg.mix_probability)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            if do_mix:
                mixed_images, style_soft = mixup_cutmix(images, style_y, cfg.mixup_alpha, cfg.cutmix_alpha, 1.0, n_style)
                _, artist_soft = mixup_cutmix(images, artist_y, cfg.mixup_alpha, cfg.cutmix_alpha, 1.0, n_artist)
                style_logits, artist_logits, emotion_logits = model(mixed_images)

                style_loss = _soft_target_ce(style_logits, style_soft)
                artist_loss = _soft_target_ce(artist_logits, artist_soft)

                emotion_loss = torch.zeros((), device=device)
            else:
                style_logits, artist_logits, emotion_logits = model(images)
                style_loss = ce_style(style_logits, style_y)
                artist_loss = ce_artist(artist_logits, artist_y)

                if emotion_mask.any():
                    emotion_loss = ce_emotion(emotion_logits[emotion_mask], emotion_y[emotion_mask])
                else:
                    emotion_loss = torch.zeros((), device=device)

            style_loss_n = style_loss / style_norm
            artist_loss_n = artist_loss / artist_norm
            emotion_loss_n = emotion_loss / emotion_norm

            use_uncertainty = bool(getattr(cfg, "use_uncertainty_weighting", True))
            if use_uncertainty:
                loss_total = (
                    0.5 * torch.exp(-model.log_var_style) * style_loss_n + 0.5 * model.log_var_style
                    + 0.5 * torch.exp(-model.log_var_artist) * artist_loss_n + 0.5 * model.log_var_artist
                    + 0.5 * torch.exp(-model.log_var_emotion) * emotion_loss_n + 0.5 * model.log_var_emotion
                )
                loss_total = loss_total.squeeze(0)
            else:
                loss_total = style_loss_n + artist_loss_n + getattr(cfg, "emotion_loss_weight", 0.7) * emotion_loss_n

            loss = loss_total / accum_steps

        if is_train:
            scaler.scale(loss).backward()
            if step % accum_steps == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                ema_inst = getattr(ModelEMA, "active_instance", None)
                if ema_inst is not None:
                    ModelEMA._allow_update = True
                    ema_inst.update(model)
                    ModelEMA._allow_update = False

        bs = images.size(0)
        total_samples += bs
        total_loss += float(loss.item()) * accum_steps * bs

        with torch.no_grad():
            style_acc = top1_accuracy(style_logits, style_y)
            artist_acc = top1_accuracy(artist_logits, artist_y)
            style_acc_sum += style_acc * bs
            artist_acc_sum += artist_acc * bs

            if (not do_mix) and emotion_mask.any():
                e_pred = emotion_logits[emotion_mask].argmax(dim=1)
                e_true = emotion_y[emotion_mask]
                emotion_correct += int((e_pred == e_true).sum().item())
                emotion_total += int(e_true.numel())

        if is_train and (step % cfg.progress_every == 0 or step == len(loader)):
            print(
                f"  [train] batch {step}/{len(loader)} | avg_loss={total_loss/max(1,total_samples):.4f} | "
                f"style={style_acc_sum/max(1,total_samples):.4f} | artist={artist_acc_sum/max(1,total_samples):.4f}"
            )

    out = {
        "loss": total_loss / max(1, total_samples),
        "style_acc": style_acc_sum / max(1, total_samples),
        "artist_acc": artist_acc_sum / max(1, total_samples),
        "emotion_acc": (emotion_correct / emotion_total) if emotion_total > 0 else float("nan"),
        "emotion_count": emotion_total,
    }
    return out


@torch.no_grad()
def evaluate_with_tta(model, loader, device, cfg, n_style, n_artist, n_emotion):
    model.eval()

    ce_style = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    ce_artist = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    emotion_ls = float(getattr(cfg, "emotion_label_smoothing", 0.12))
    ce_emotion = nn.CrossEntropyLoss(label_smoothing=emotion_ls)

    total_loss = 0.0
    total_samples = 0

    style_acc_sum = 0.0
    artist_acc_sum = 0.0
    emotion_correct = 0
    emotion_total = 0

    style_norm = max(1.0, math.log(max(2, n_style)))
    artist_norm = max(1.0, math.log(max(2, n_artist)))
    emotion_norm = max(1.0, math.log(max(2, n_emotion)))

    for batch in loader:
        images, style_y, artist_y, emotion_y, emotion_mask = batch
        images = images.to(device, non_blocking=True)
        style_y = style_y.to(device, non_blocking=True)
        artist_y = artist_y.to(device, non_blocking=True)
        emotion_y = emotion_y.to(device, non_blocking=True)
        emotion_mask = emotion_mask.to(device, non_blocking=True).bool()

        style_logits, artist_logits, emotion_logits = model(images)
        if cfg.tta:
            s2, a2, e2 = model(torch.flip(images, dims=[3]))
            style_logits = (style_logits + s2) / 2.0
            artist_logits = (artist_logits + a2) / 2.0
            emotion_logits = (emotion_logits + e2) / 2.0

        style_loss = ce_style(style_logits, style_y)
        artist_loss = ce_artist(artist_logits, artist_y)
        if emotion_mask.any():
            emotion_loss = ce_emotion(emotion_logits[emotion_mask], emotion_y[emotion_mask])
        else:
            emotion_loss = torch.zeros((), device=device)

        loss = (style_loss / style_norm) + (artist_loss / artist_norm) + getattr(cfg, "emotion_loss_weight", 0.7) * (emotion_loss / emotion_norm)

        bs = images.size(0)
        total_samples += bs
        total_loss += float(loss.item()) * bs

        style_acc_sum += top1_accuracy(style_logits, style_y) * bs
        artist_acc_sum += top1_accuracy(artist_logits, artist_y) * bs

        if emotion_mask.any():
            e_pred = emotion_logits[emotion_mask].argmax(dim=1)
            e_true = emotion_y[emotion_mask]
            emotion_correct += int((e_pred == e_true).sum().item())
            emotion_total += int(e_true.numel())

    return {
        "loss": total_loss / max(1, total_samples),
        "style_acc": style_acc_sum / max(1, total_samples),
        "artist_acc": artist_acc_sum / max(1, total_samples),
        "emotion_acc": (emotion_correct / emotion_total) if emotion_total > 0 else float("nan"),
        "emotion_count": emotion_total,
    }


print("Test 9 patch active: fixed emotion mixup handling, smoothed emotion loss, normalized task losses, MLP heads, uncertainty weighting, and per-step EMA updates.")

In [ ]:
# --- LLRD patch (Layer-wise Learning Rate Decay) ---
# ViT fine-tuning trick: lower LR for earlier blocks, higher for later blocks.

def _vit_layer_id(name: str, num_blocks: int) -> int:
    if name.startswith("backbone.blocks."):
        parts = name.split(".")
        if len(parts) > 2 and parts[2].isdigit():
            return int(parts[2]) + 1
    if name.startswith("backbone.patch_embed") or name in {"backbone.cls_token", "backbone.pos_embed"}:
        return 0
    if name.startswith("backbone.norm") or name.startswith("backbone.fc_norm") or name.startswith("backbone.pre_logits"):
        return num_blocks + 1
    return num_blocks + 1


def get_trainable_params(model: MultiTaskModel, backbone_lr: float, head_lr: float, wd: float):
    cfg_obj = globals().get("cfg", None)
    use_llrd = bool(getattr(model, "use_llrd", getattr(cfg_obj, "use_llrd", True)))
    llrd_decay = float(getattr(model, "llrd_decay", getattr(cfg_obj, "llrd_decay", 0.9)))

    num_blocks = len(getattr(model.backbone, "blocks", [])) if hasattr(model.backbone, "blocks") else 0

    head_params = [
        p for p in list(model.style_head.parameters()) + list(model.artist_head.parameters()) + list(model.emotion_head.parameters())
        if p.requires_grad
    ]
    uncertainty = [model.log_var_style, model.log_var_artist, model.log_var_emotion]

    groups = []

    if use_llrd and num_blocks > 0:
        layer_buckets = {}
        max_layer = num_blocks + 1

        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if not name.startswith("backbone."):
                continue
            layer_id = _vit_layer_id(name, num_blocks)
            scale = llrd_decay ** (max_layer - layer_id)
            if scale not in layer_buckets:
                layer_buckets[scale] = []
            layer_buckets[scale].append(p)

        for scale, params in sorted(layer_buckets.items(), key=lambda x: x[0]):
            groups.append({"params": params, "lr": backbone_lr * scale, "weight_decay": wd})
    else:
        backbone = [p for p in model.backbone.parameters() if p.requires_grad]
        if backbone:
            groups.append({"params": backbone, "lr": backbone_lr, "weight_decay": wd})

    groups.append({"params": head_params + uncertainty, "lr": head_lr, "weight_decay": wd})
    return groups


print("LLRD patch active: backbone uses layer-wise LR decay (default decay=0.9).")

In [ ]:
cfg = TrainConfig(
    model_name="vit_base_patch16_384.augreg_in21k_ft_in1k",
    image_size=384,
    batch_size=8,
    effective_batch_size=64,
    head_epochs=4,
    ft_epochs=56,
    patience=16,
    warmup_epochs=4,
    head_lr=4e-4,
    ft_backbone_lr=4e-6,
    ft_head_lr=1.8e-5,
    weight_decay=1.2e-4,
    mixup_alpha=0.6,
    cutmix_alpha=1.0,
    mix_probability=0.9,
    label_smoothing=0.08,
    ema_decay=0.99985,
    grad_clip_norm=1.0,
    tta=True,
    use_weighted_sampler=True,
    seed=42,
    num_workers=0,
    progress_every=300,
    emotion_loss_weight=0.7,
)

# Patch knobs
cfg.emotion_label_smoothing = 0.12
cfg.use_uncertainty_weighting = True

# LLRD knobs (can slightly improve ViT fine-tuning)
cfg.use_llrd = True
cfg.llrd_decay = 0.9

# If you hit CUDA OOM, reduce batch_size to 6 first.

In [ ]:
history_df, final_val, final_test = fit(Path.cwd(), cfg)

In [ ]:
print("History rows:", len(history_df))
if len(history_df) > 0:
    best_idx = history_df["val_combined_score"].idxmax()
    print("\nBest validation row (combined score):")
    display(history_df.loc[[best_idx]])

print("\nFinal Validation:", final_val)
print("Final Test:", final_test)

results_dir = Path.cwd() / "models" / "results"
print("\nSaved files:")
print("-", results_dir / "wikiart_test9_multitask_history.csv")
print("-", results_dir / "wikiart_tests_1_to_9_summary.csv")
print("-", Path.cwd() / "models" / "wikiart_test9_multitask_best.pt")